# 03 — Modeling

Regression and classification models for the Ames Housing dataset.  
Reads preprocessed data from `../data/processed/`.

**Scope:** Modeling and evaluation only. No preprocessing, feature engineering, or EDA.

**Prerequisite:** Run `02_preprocessing.ipynb` first to generate the processed CSVs.

## 1. Imports

In [32]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_squared_error,
    roc_auc_score,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 2. Load Processed Data

In [33]:
train_model = pd.read_csv('../data/processed/train.csv')
val_model   = pd.read_csv('../data/processed/val.csv')
test_model  = pd.read_csv('../data/processed/test.csv')

print(f"Train:      {train_model.shape}")
print(f"Validation: {val_model.shape}")
print(f"Test:       {test_model.shape}")

Train:      (2051, 95)
Validation: (439, 95)
Test:       (440, 95)


---
# Part A — Regression Models

Target: `log_SalePrice` (log-transformed to address right skew).  
Evaluation: RMSE on log scale and original scale.

## Model 1 — Baseline (Overall Qual + Total_SF + House_Age)

In [34]:
model1 = smf.ols(
    'log_SalePrice ~ Q("Overall Qual") + Total_SF + House_Age',
    data=train_model
).fit()
print(model1.summary())

val_model['pred_model1'] = model1.predict(val_model)
rmse_model1 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model1']))
print(f"\nModel 1 — RMSE (log scale): {rmse_model1:.4f}")

val_model['pred_model1_original'] = np.exp(val_model['pred_model1'])
rmse_model1_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model1_original']))
print(f"Model 1 — RMSE (original scale): ${rmse_model1_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     2864.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:49   Log-Likelihood:                 631.55
No. Observations:                2051   AIC:                            -1255.
Df Residuals:                    2047   BIC:                            -1233.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            10.8464      0.02

## Model 2 — Multiple Numeric Predictors

In [35]:
formula_model2 = '''
log_SalePrice ~ Qual_Cond
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
'''

model2 = smf.ols(formula_model2, data=train_model).fit()
print(model2.summary())

val_model['pred_model2'] = model2.predict(val_model)
rmse_model2 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model2']))
print(f"\nModel 2 — RMSE (log scale): {rmse_model2:.4f}")

val_model['pred_model2_original'] = np.exp(val_model['pred_model2'])
rmse_model2_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model2_original']))
print(f"Model 2 — RMSE (original scale): ${rmse_model2_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.863
Model:                            OLS   Adj. R-squared:                  0.863
Method:                 Least Squares   F-statistic:                     1844.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:49   Log-Likelihood:                 982.41
No. Observations:                2051   AIC:                            -1949.
Df Residuals:                    2043   BIC:                            -1904.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [36]:
# VIF — Model 2
X_model2 = pd.DataFrame({
    'Qual_Cond':    train_model['Qual_Cond'],
    'log_Total_SF':    np.log(train_model['Total_SF']),
    'log_Lot_Area':    np.log(train_model['Lot Area']),
    'log_Garage_Area': np.log(train_model['Garage Area'] + 1),
    'Total_Bath':      train_model['Total_Bath'],
    'House_Age':       train_model['House_Age'],
    'Years_Since_Remod': train_model['Years_Since_Remod']
}).dropna()

X_model2 = sm.add_constant(X_model2)

vif_model2 = pd.DataFrame({
    'variable': X_model2.columns,
    'VIF': [variance_inflation_factor(X_model2.values, i) for i in range(X_model2.shape[1])]
})
print("VIF — Model 2:")
vif_model2[vif_model2['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 2:


,variable,VIF
2,log_Total_SF,2.147234
6,House_Age,2.098229
7,Years_Since_Remod,2.088388
5,Total_Bath,1.934484
1,Qual_Cond,1.541421
3,log_Lot_Area,1.265321
4,log_Garage_Area,1.233064


## Model 3 — Numeric + Categorical Predictors (Neighborhood_grouped)

In [37]:
formula_model3 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_grouped)
                + C(Q("Bldg Type"))
                + C(Q("Kitchen Qual"))
'''

model3 = smf.ols(formula_model3, data=train_model).fit()
print(model3.summary())

val_model['pred_model3'] = model3.predict(val_model)
rmse_model3 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model3']))
print(f"\nModel 3 — RMSE (log scale): {rmse_model3:.4f}")

val_model['pred_model3_original'] = np.exp(val_model['pred_model3'])
rmse_model3_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model3_original']))
print(f"Model 3 — RMSE (original scale): ${rmse_model3_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.875
Model:                            OLS   Adj. R-squared:                  0.874
Method:                 Least Squares   F-statistic:                     617.3
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:50   Log-Likelihood:                 1074.4
No. Observations:                2051   AIC:                            -2101.
Df Residuals:                    2027   BIC:                            -1966.
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## Model 4 — Simplified Categorical Predictors

In [38]:
formula_model4 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
'''

model4 = smf.ols(formula_model4, data=train_model).fit()
print(model4.summary())

val_model['pred_model4'] = model4.predict(val_model)
rmse_model4 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model4']))
print(f"\nModel 4 — RMSE (log scale): {rmse_model4:.4f}")

val_model['pred_model4_original'] = np.exp(val_model['pred_model4'])
rmse_model4_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model4_original']))
print(f"Model 4 — RMSE (original scale): ${rmse_model4_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.868
Model:                            OLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     1217.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:50   Log-Likelihood:                 1016.8
No. Observations:                2051   AIC:                            -2010.
Df Residuals:                    2039   BIC:                            -1942.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [39]:
# VIF — Model 4 (continuous features only)
X_model4 = pd.DataFrame({
    'Overall Qual':    train_model['Overall Qual'],
    'log_Total_SF':    np.log(train_model['Total_SF']),
    'log_Lot_Area':    np.log(train_model['Lot Area']),
    'log_Garage_Area': np.log(train_model['Garage Area'] + 1),
    'Total_Bath':      train_model['Total_Bath'],
    'House_Age':       train_model['House_Age'],
    'Years_Since_Remod': train_model['Years_Since_Remod']
}).dropna()

X_model4 = sm.add_constant(X_model4)

vif_model4 = pd.DataFrame({
    'variable': X_model4.columns,
    'VIF': [variance_inflation_factor(X_model4.values, i) for i in range(X_model4.shape[1])]
})
print("VIF — Model 4:")
vif_model4[vif_model4['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 4:


,variable,VIF
1,Overall Qual,2.684269
2,log_Total_SF,2.636224
6,House_Age,2.115679
5,Total_Bath,1.935169
7,Years_Since_Remod,1.840253
3,log_Lot_Area,1.280385
4,log_Garage_Area,1.228256


## Model 5 — Interaction: Neighborhood × Total_SF

In [40]:
formula_model5 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
                + C(Neighborhood_simple):np.log(Total_SF)
'''

model5 = smf.ols(formula_model5, data=train_model).fit()
print(model5.summary())

val_model['pred_model5'] = model5.predict(val_model)
rmse_model5 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model5']))
print(f"\nModel 5 — RMSE (log scale): {rmse_model5:.4f}")

val_model['pred_model5_original'] = np.exp(val_model['pred_model5'])
rmse_model5_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model5_original']))
print(f"Model 5 — RMSE (original scale): ${rmse_model5_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.870
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     1139.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:51   Log-Likelihood:                 1035.8
No. Observations:                2051   AIC:                            -2046.
Df Residuals:                    2038   BIC:                            -1973.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------

In [41]:
# VIF — Model 5 (full design matrix including categoricals and interaction)
X = model5.model.exog
feature_names = model5.model.exog_names

vif_model5 = pd.DataFrame({
    'variable': feature_names,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
print("VIF — Model 5:")
vif_model5[vif_model5['variable'] != 'Intercept'].sort_values('VIF', ascending=False)

VIF — Model 5:


,variable,VIF
1,C(Neighborhood_simple)[T.Other],1169.623745
7,C(Neighborhood_simple)[T.Other]:np.log(Total_SF),1109.215736
6,np.log(Total_SF),17.564145
5,"Q(""Overall Qual"")",3.036688
11,House_Age,2.209867
12,Years_Since_Remod,2.148171
10,Total_Bath,2.039225
3,C(Kitchen_Qual_grouped)[T.Medium],2.019583
8,"np.log(Q(""Lot Area""))",1.863484
4,C(Bldg_Type_simple)[T.Other],1.613305


## Model 6 — Reduced Core Model

In [42]:
formula_model6 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
'''

model6 = smf.ols(formula_model6, data=train_model).fit()
print(model6.summary())

val_model['pred_model6'] = model6.predict(val_model)
rmse_model6 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model6']))
print(f"\nModel 6 — RMSE (log scale): {rmse_model6:.4f}")

val_model['pred_model6_original'] = np.exp(val_model['pred_model6'])
rmse_model6_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model6_original']))
print(f"Model 6 — RMSE (original scale): ${rmse_model6_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     1833.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:51   Log-Likelihood:                 686.27
No. Observations:                2051   AIC:                            -1361.
Df Residuals:                    2045   BIC:                            -1327.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [43]:
# VIF — Model 6 (continuous features only)
X_model6 = pd.DataFrame({
    'Overall_Qual': train_model['Overall Qual'],
    'log_Total_SF': np.log(train_model['Total_SF'])
}).dropna()

X_model6 = sm.add_constant(X_model6)

vif_model6 = pd.DataFrame({
    'variable': X_model6.columns,
    'VIF': [variance_inflation_factor(X_model6.values, i) for i in range(X_model6.shape[1])]
})
print("VIF — Model 6:")
vif_model6[vif_model6['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 6:


,variable,VIF
2,log_Total_SF,1.82995
1,Overall_Qual,1.82995


In [44]:
# =========================================================
# Modelo 7: substituição de Overall Qual por Qual_Cond
# =========================================================

formula_model7 = '''
log_SalePrice ~ Qual_Cond
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
'''

model7 = smf.ols(formula_model7, data=train_model).fit()
print(model7.summary())

# --- Avaliação ---
val_model['pred_model7'] = model7.predict(val_model)

rmse_model7 = np.sqrt(mean_squared_error(
    val_model['log_SalePrice'],
    val_model['pred_model7']
))
print(f"Model 7 - RMSE (log scale): {rmse_model7:.4f}")

val_model['pred_model7_original'] = np.exp(val_model['pred_model7'])

rmse_model7_original = np.sqrt(mean_squared_error(
    val_model['SalePrice'],
    val_model['pred_model7_original']
))
print(f"Model 7 - RMSE (original scale): ${rmse_model7_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.872
Model:                            OLS   Adj. R-squared:                  0.871
Method:                 Least Squares   F-statistic:                     1260.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:32:52   Log-Likelihood:                 1047.6
No. Observations:                2051   AIC:                            -2071.
Df Residuals:                    2039   BIC:                            -2004.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

## Regression Summary

In [45]:
regression_summary = pd.DataFrame([
    {'Model': 'Model 1', 'Description': 'Baseline (Qual + SF + Age)',           'RMSE_log': rmse_model1, 'RMSE_original': rmse_model1_original},
    {'Model': 'Model 2', 'Description': 'Numeric predictors',                   'RMSE_log': rmse_model2, 'RMSE_original': rmse_model2_original},
    {'Model': 'Model 3', 'Description': '+ Categorical (grouped neighborhood)', 'RMSE_log': rmse_model3, 'RMSE_original': rmse_model3_original},
    {'Model': 'Model 4', 'Description': '+ Simplified categoricals',            'RMSE_log': rmse_model4, 'RMSE_original': rmse_model4_original},
    {'Model': 'Model 5', 'Description': '+ Interaction Neighborhood x SF',      'RMSE_log': rmse_model5, 'RMSE_original': rmse_model5_original},
    {'Model': 'Model 6', 'Description': 'Reduced core model',                   'RMSE_log': rmse_model6, 'RMSE_original': rmse_model6_original},
    {'Model': 'Model 7', 'Description': 'Qual_Cond instead of Overall Qual',    'RMSE_log': rmse_model7, 'RMSE_original': rmse_model7_original},
])

regression_summary

,Model,Description,RMSE_log,RMSE_original
0,Model 1,Baseline (Qual + SF + Age),0.191594,29528.708378
1,Model 2,Numeric predictors,0.162772,26655.853365
2,Model 3,+ Categorical (grouped neighborhood),0.164843,24209.609251
3,Model 4,+ Simplified categoricals,0.170060,26006.760512
4,Model 5,+ Interaction Neighborhood x SF,0.170082,25827.299230
5,Model 6,Reduced core model,0.196269,29583.602664
6,Model 7,Qual_Cond instead of Overall Qual,0.161416,26031.042095


---
# Part B — Classification Models

Target: `is_high` (binary — top price tier vs rest).  
Evaluation: confusion matrix, classification report, ROC-AUC. Threshold: 0.4.

## Logit 1 — Baseline (Overall Qual + Total_SF)

In [46]:
formula_logit1 = 'is_high ~ Q("Overall Qual") + np.log(Total_SF)'

model_logit1 = smf.logit(formula_logit1, data=train_model).fit()
print(model_logit1.summary())

val_model['pred_prob_1']  = model_logit1.predict(val_model)
val_model['pred_class_1'] = (val_model['pred_prob_1'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_1']))
print(classification_report(val_model['is_high'], val_model['pred_class_1']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_1']):.4f}")

Optimization terminated successfully.
         Current function value: 0.248732
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2048
Method:                           MLE   Df Model:                            2
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6076
Time:                        19:32:52   Log-Likelihood:                -510.15
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -74.4692      4.230    -17.605      0.000     -82.760     -66.178
Q("Overa

## Logit 2 — Baseline + Neighborhood + Kitchen Qual

In [47]:
formula_logit2 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit2 = smf.logit(formula_logit2, data=train_model).fit()
print(model_logit2.summary())

val_model['pred_prob_2']  = model_logit2.predict(val_model)
val_model['pred_class_2'] = (val_model['pred_prob_2'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_2']))
print(classification_report(val_model['is_high'], val_model['pred_class_2']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_2']):.4f}")

Optimization terminated successfully.
         Current function value: 0.224823
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2045
Method:                           MLE   Df Model:                            5
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6453
Time:                        19:32:53   Log-Likelihood:                -461.11
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -75.6781      4.511    -16

## Logit 3 — Extended Numeric + Categorical

In [48]:
formula_logit3 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + np.log(Q("Garage Area") + 1)
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit3 = smf.logit(formula_logit3, data=train_model).fit()
print(model_logit3.summary())

val_model['pred_prob_3']  = model_logit3.predict(val_model)
val_model['pred_class_3'] = (val_model['pred_prob_3'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_3']))
print(classification_report(val_model['is_high'], val_model['pred_class_3']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_3']):.4f}")

Optimization terminated successfully.
         Current function value: 0.200842
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2040
Method:                           MLE   Df Model:                           10
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6832
Time:                        19:32:53   Log-Likelihood:                -411.93
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -80.1633      5.023    -15

In [49]:
# Odds ratios — Logit 3
odds_ratios_3 = pd.DataFrame({
    'variable':   model_logit3.params.index,
    'coef':       model_logit3.params.values,
    'odds_ratio': np.exp(model_logit3.params.values)
})
print("Odds Ratios — Logit 3:")
odds_ratios_3

Odds Ratios — Logit 3:


,variable,coef,odds_ratio
0,Intercept,-80.163285,1.532948e-35
1,C(Neighborhood_simple)[T.Other],-1.133242,3.219878e-01
2,C(Kitchen_Qual_grouped)[T.Low],-0.933068,3.933450e-01
3,C(Kitchen_Qual_grouped)[T.Medium],-1.174145,3.090831e-01
4,"Q(""Overall Qual"")",1.001918,2.723500e+00
5,np.log(Total_SF),7.469833,1.754314e+03
6,"np.log(Q(""Lot Area""))",1.352607,3.867497e+00
7,"np.log(Q(""Garage Area"") + 1)",0.360111,1.433488e+00
8,Total_Bath,0.589171,1.802493e+00
9,House_Age,-0.007293,9.927332e-01


## Logit 4 — Parsimonious (removes weak predictors from Logit 3)

In [50]:
formula_logit4 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit4 = smf.logit(formula_logit4, data=train_model).fit()
print(model_logit4.summary())

val_model['pred_prob_4']  = model_logit4.predict(val_model)
val_model['pred_class_4'] = (val_model['pred_prob_4'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_4']))
print(classification_report(val_model['is_high'], val_model['pred_class_4']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_4']):.4f}")

Optimization terminated successfully.
         Current function value: 0.202067
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2041
Method:                           MLE   Df Model:                            9
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6812
Time:                        19:32:53   Log-Likelihood:                -414.44
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -79.7062      4.990    -15

In [51]:
# Odds ratios — Logit 4
odds_ratios_4 = pd.DataFrame({
    'variable':   model_logit4.params.index,
    'coef':       model_logit4.params.values,
    'odds_ratio': np.exp(model_logit4.params.values)
})
print("Odds Ratios — Logit 4:")
odds_ratios_4

Odds Ratios — Logit 4:


,variable,coef,odds_ratio
0,Intercept,-79.706179,2.421288e-35
1,C(Neighborhood_simple)[T.Other],-1.183623,3.061676e-01
2,C(Kitchen_Qual_grouped)[T.Low],-1.060745,3.461980e-01
3,C(Kitchen_Qual_grouped)[T.Medium],-1.186237,3.053682e-01
4,"Q(""Overall Qual"")",1.033825,2.811799e+00
5,np.log(Total_SF),7.599138,1.996475e+03
6,"np.log(Q(""Lot Area""))",1.425836,4.161337e+00
7,Total_Bath,0.570108,1.768458e+00
8,House_Age,-0.009171,9.908707e-01
9,Years_Since_Remod,-0.017991,9.821695e-01


## Logit Final — Without Kitchen Qual

In [52]:
formula_logit_final = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
'''

model_logit_final = smf.logit(formula_logit_final, data=train_model).fit()
print(model_logit_final.summary())

val_model['pred_prob_final']  = model_logit_final.predict(val_model)
val_model['pred_class_final'] = (val_model['pred_prob_final'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_final']))
print(classification_report(val_model['is_high'], val_model['pred_class_final']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_final']):.4f}")

Optimization terminated successfully.
         Current function value: 0.208128
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2043
Method:                           MLE   Df Model:                            7
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6717
Time:                        19:32:54   Log-Likelihood:                -426.87
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                         -79.8841      4.906    -16.283  

## Classification Summary

In [53]:
classification_summary = pd.DataFrame([
    {'Model': 'Logit 1', 'Description': 'Baseline (Qual + SF)',                        'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_1'])},
    {'Model': 'Logit 2', 'Description': '+ Neighborhood + Kitchen Qual',               'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_2'])},
    {'Model': 'Logit 3', 'Description': '+ Extended numerics + categoricals',          'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_3'])},
    {'Model': 'Logit 4', 'Description': 'Parsimonious (removes weak from Logit 3)',   'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_4'])},
    {'Model': 'Logit Final', 'Description': 'Without Kitchen Qual',                   'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_final'])},
])

classification_summary

,Model,Description,ROC-AUC
0,Logit 1,Baseline (Qual + SF),0.960671
1,Logit 2,+ Neighborhood + Kitchen Qual,0.963993
2,Logit 3,+ Extended numerics + categoricals,0.975012
3,Logit 4,Parsimonious (removes weak from Logit 3),0.975060
4,Logit Final,Without Kitchen Qual,0.975084


In [54]:
base_formula = '''
log_SalePrice ~  np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_grouped)
                + C(Q("Bldg Type"))
                + C(Q("Kitchen Qual"))
'''

In [55]:
candidate_features = [
    "Qual_Cond",
    "Has_Garage"
]

results = []

# --- baseline ---
model = smf.ols(base_formula, data=train_model).fit()

preds = model.predict(val_model)

rmse_log = np.sqrt(mean_squared_error(val_model['log_SalePrice'], preds))
rmse_original = np.sqrt(mean_squared_error(
    val_model['SalePrice'], np.exp(preds)
))

results.append({
    "Feature": "BASELINE",
    "RMSE_log": rmse_log,
    "RMSE_original": rmse_original
})


# --- testes ---
for feat in candidate_features:
    formula_test = base_formula + f" + {feat}"

    model = smf.ols(formula_test, data=train_model).fit()

    preds = model.predict(val_model)

    rmse_log = np.sqrt(mean_squared_error(val_model['log_SalePrice'], preds))
    rmse_original = np.sqrt(mean_squared_error(
        val_model['SalePrice'], np.exp(preds)
    ))

    results.append({
        "Feature": feat,
        "RMSE_log": rmse_log,
        "RMSE_original": rmse_original
    })

pd.DataFrame(results).sort_values("RMSE_original")

,Feature,RMSE_log,RMSE_original
1,Qual_Cond,0.155896,23579.116696
2,Has_Garage,0.185523,28448.518826
0,BASELINE,0.185624,28703.798071


In [56]:
test_model['pred_log'] = model4.predict(test_model)

# RMSE log
rmse_log_test = np.sqrt(mean_squared_error(
    test_model['log_SalePrice'],
    test_model['pred_log']
))

# RMSE original
test_model['pred_original'] = np.exp(test_model['pred_log'])

rmse_original_test = np.sqrt(mean_squared_error(
    test_model['SalePrice'],
    test_model['pred_original']
))

print(f"Test RMSE (log): {rmse_log_test:.4f}")
print(f"Test RMSE (original): ${rmse_original_test:,.0f}")

Test RMSE (log): 0.1546
Test RMSE (original): $29,476


In [57]:
test_model['pred_prob'] = model_logit_final.predict(test_model)

In [58]:
threshold = 0.4  # ou o que escolheste

test_model['pred_class'] = (test_model['pred_prob'] >= threshold).astype(int)

In [59]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(test_model['is_high'], test_model['pred_class']))

print(classification_report(test_model['is_high'], test_model['pred_class']))

roc = roc_auc_score(test_model['is_high'], test_model['pred_prob'])
print(f"ROC-AUC: {roc:.4f}")

[[264  22]
 [ 10 144]]
              precision    recall  f1-score   support

           0       0.96      0.92      0.94       286
           1       0.87      0.94      0.90       154

    accuracy                           0.93       440
   macro avg       0.92      0.93      0.92       440
weighted avg       0.93      0.93      0.93       440

ROC-AUC: 0.9825
